In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# token = user_secrets.get_secret("HF_TOKEN")
import os
from huggingface_hub import login

# Get the token from Kaggle secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Log in to Hugging Face
login(token=hf_token)

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas openpyxl


In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=False,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4"
# )

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# # model = AutoModelForCausalLM.from_pretrained(
# #     model_name,
# #     quantization_config=bnb_config,
# #     device_map="auto"
# # )
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.float32,
#     device_map="auto"
# )

# model.eval()


In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4"
# )

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto"
# )

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # VERY IMPORTANT
    device_map="auto"            # splits across GPUs
)

model.eval()


In [ ]:
import pandas as pd

df = pd.read_excel("/kaggle/input/datasets/jishnumsc24/eng-test-im/output_1 - Copy.xlsx")  # change path

print(df.head())


In [ ]:
df.columns

In [ ]:
df = df.rename(columns={'Unnamed: 1': 'ocr'})

In [ ]:
df = df.drop(columns=['text'])

In [ ]:
def build_prompt(ocr_text, analysis_text):
    return f"""
You are a strict classification model.

Task:
Classify the meme into ONLY ONE of the following classes:

HOMOPHOBIA
TRANSPHOBIA
NON_ANTI_LGBTQ

Rules:
- Output only ONE WORD, that is one of the classes, the class name should be ALL CAPS.
- No explanation.
- No punctuation.
- No extra text.

OCR: {ocr_text}

ANALYSIS: {analysis_text}

Answer:
"""


In [ ]:
def classify_sample(ocr_text, analysis_text):
    prompt = build_prompt(ocr_text, analysis_text)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            temperature=0.0
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract only the last line
    prediction = response.split("Answer:")[-1].strip().split("\n")[0].strip()

    return prediction


In [ ]:
from tqdm import tqdm
predictions = []

for idx, row in tqdm(df.iterrows()):
    pred = classify_sample(row["ocr"], row["analysis"])
    predictions.append(pred)

df["prediction"] = predictions


In [ ]:
df["prediction"].unique()

In [ ]:
df.to_csv('output.csv')

In [ ]:
# valid_labels = ["HOMOPHOBIA", "TRANSPHOBIA", "NON_ANTI_LGBTQ"]

# def clean_label(label):
#     label = label.strip().upper()
#     if label in valid_labels:
#         return label
#     else:
#         return "NON_ANTI_LGBTQ"

# df["prediction"] = df["prediction"].apply(clean_label)
